# MacroRegimes3 — Part 2: Tree-Based Macroeconomic Regime Classification

Based on:
> Bie, Diebold, He, Li. *Machine Learning and the Yield Curve: Tree-Based Macroeconomic Regime Switching* (2024).

A sequential greedy tree splits the sample into 3 regimes. Candidate splits are (quantile-standardized variable × threshold);
the splitting criterion is the Bayesian marginal likelihood of the Svensson-reconstructed yields (13 maturities) under a
Normal-Inverse-Gamma conjugate model. Split out of `MacroRegimes2.ipynb`, Sections 4–10 and 13.0–13.1.

**Inputs** (from `Data.ipynb`): `MacroRegimes3_Data_Quantiles.csv`, `_Target.csv`, `_Levels.csv`, `_Recession.csv`, `_Columns.csv`.

**Outputs:**

| File | Contents | Used by |
|---|---|---|
| `MacroRegimes3_Tree_Labels.csv` | full-sample regime per month (raw and canonical labels) | PV-RF, HMM |
| `MacroRegimes3_Tree_Results.csv` | levels + quantiles + regime per month (same layout as MacroRegimes2) | — |
| `MacroRegimes3_Tree_Splits.csv` | the two splits of the full-sample tree | — |
| `MacroRegimes3_Tree_FoldLabels.csv` | regime labels from every evaluation tree and local training tree, per test year | PV-RF (OOS evaluation) |
| `MacroRegimes3_Tree_FoldSplits.csv` | the splits chosen by every fold tree | PV-RF, diagnostics |

The fold trees (Section 7) are what the PV-RF notebook needs for its expanding-window evaluation; growing them here
means the PV-RF notebook never has to grow a tree.

## 0. Imports and User Inputs

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import gammaln

# Folder holding every CSV input and output (locally: DataAndResults/ next to the MacroRegimes3/ notebook folder)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_folder = '/content/drive/MyDrive/DataAndResults'
except ImportError:
    data_folder = os.path.join('..', 'DataAndResults')

In [ ]:
# Tree parameters
thresholds = [0.2, 0.4, 0.6, 0.8]  # candidate split thresholds on the quantile scale (paper pg. 17)
min_regime_months = 24             # minimum observations per regime (pg. 11)
max_regimes = 3                    # 2 splits (pg. 11)

# Expanding-window fold parameters (must match what PVRFClassifier.ipynb expects)
min_training_years = 5  # first test year starts 5 years after the sample start
gap_months = 1          # embargo month between training end and test start

# Regime colours used throughout the notebooks
regime_colors = {0: 'coral', 1: 'steelblue', 2: 'mediumseagreen', 3: 'orchid', 4: 'goldenrod'}

In [ ]:
# Load the data built by Data.ipynb
data_q    = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Quantiles.csv', index_col='date', parse_dates=True)
target    = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Target.csv', index_col='date', parse_dates=True)
data      = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Levels.csv', index_col='date', parse_dates=True)
recession = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Recession.csv', index_col='date', parse_dates=True)
columns   = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Columns.csv')

common_idx = data_q.index
all_cols = columns['variable'].tolist()
bond_cols = columns.loc[columns['group'] == 'bond', 'variable'].tolist()
yield_cols = [c for c in target.columns if c.startswith('Y')]  # Y3M ... Y120M
yields_svensson = target[yield_cols]
target_data = yields_svensson.values

recession_aligned = recession.reindex(common_idx, method='ffill').fillna(0)
rec_mask = recession_aligned['USREC'].values == 1

print(f'Split candidates: {data_q.shape} | Target: {target_data.shape}')
print(f'Sample: {common_idx.min().date()} → {common_idx.max().date()}')

## 1. Bayesian Marginal Likelihood (Splitting Criterion)

Instead of the marginal likelihood of a DNS model (as in the paper) we use the closed-form marginal likelihood of a
Normal-Inverse-Gamma conjugate model, which penalizes splits that shrink regimes without improving fit.

**Model:** $Y_i \sim \mathcal{N}(\mu, \sigma^2)$, $\mu \sim \mathcal{N}(\mu_0, \sigma^2/\kappa_0)$, $\sigma^2 \sim \text{IG}(\alpha_0, \beta_0)$.

**Posterior hyperparameters:** $\kappa_n = \kappa_0 + n$, $\mu_n = \frac{\kappa_0\mu_0 + n\bar y}{\kappa_n}$, $\alpha_n = \alpha_0 + n/2$,
$\beta_n = \beta_0 + \tfrac12\sum(Y_i-\bar y)^2 + \frac{n\kappa_0}{\kappa_0+n}\frac{(\mu_0-\bar y)^2}{2}$.

**Marginal likelihood:** $p(Y) = (2\pi)^{-n/2}\sqrt{\kappa_0/\kappa_n}\,\frac{\Gamma(\alpha_n)}{\Gamma(\alpha_0)}\,\frac{\beta_0^{\alpha_0}}{\beta_n^{\alpha_n}}$.

> TODO (known issue, not fixed yet): the 13 maturities are treated as independent i.i.d. Normals, which overcounts evidence
> (adjacent yields are ~0.99 correlated) and makes the tree a *level* clusterer. See the critique notes.

In [ ]:
def log_marginal_likelihood_nig(Y, mu0=0.0, kappa0=1.0, alpha0=1.0, beta0=1.0):
    """Log marginal likelihood of a 1-D sample under the Normal-Inverse-Gamma model."""
    Y = np.asarray(Y, dtype=float)
    Y = Y[np.isfinite(Y)]
    n = len(Y)
    if n < 2:
        return -np.inf  # a regime needs at least 2 points to define a variance

    ybar = Y.mean()
    kappa_n = kappa0 + n
    alpha_n = alpha0 + n / 2.0
    beta_n = beta0 + np.sum((Y - ybar) ** 2) / 2.0 + (n * kappa0 * (ybar - mu0) ** 2) / (2.0 * kappa_n)

    return (-(n / 2.0) * np.log(2.0 * np.pi) + 0.5 * np.log(kappa0 / kappa_n)
            + gammaln(alpha_n) - gammaln(alpha0) + alpha0 * np.log(beta0) - alpha_n * np.log(beta_n))

def total_log_ml(regime_labels, target_data):
    """Sum of log marginal likelihoods over all regimes and all target columns."""
    regime_labels = np.asarray(regime_labels)
    target = np.asarray(target_data)
    if target.ndim == 1:
        target = target.reshape(-1, 1)

    total = 0.0
    for label in np.unique(regime_labels):
        mask = regime_labels == label
        if mask.sum() < 2:
            return -np.inf
        for j in range(target.shape[1]):
            total += log_marginal_likelihood_nig(target[mask, j])
    return total

## 2. Sequential Tree Growing
Greedy search: at each step, evaluate every (variable × threshold) on every current leaf and apply the split with the
highest total marginal likelihood (paper sec. 2.2.2).

In [ ]:
def find_best_split(regime_labels, leaf_label, data_q, target_data, thresholds, min_obs, next_label, verbose=True):
    """Search all (variable, threshold) combinations for the best split of one leaf."""
    leaf_mask = regime_labels == leaf_label
    best = {'ml': -np.inf, 'macro': None, 'threshold': None, 'labels': None}
    all_results = []

    for macro in data_q.columns:
        values = data_q[macro].values
        for threshold in thresholds:
            right = leaf_mask & (values >= threshold)
            left = leaf_mask & (values < threshold)
            if left.sum() < min_obs or right.sum() < min_obs:
                continue

            candidate_labels = regime_labels.copy()
            candidate_labels[right] = next_label
            ml = total_log_ml(candidate_labels, target_data)
            all_results.append({'macro': macro, 'threshold': threshold, 'ml': ml})

            if ml > best['ml']:
                best = {'ml': ml, 'macro': macro, 'threshold': threshold, 'labels': candidate_labels}

    if verbose:
        print(f'\n  Top 5 candidates for Leaf {leaf_label}:')
        for r in sorted(all_results, key=lambda x: x['ml'], reverse=True)[:5]:
            print(f'    {r["macro"]:20s} threshold = {r["threshold"]} | log ML = {r["ml"]:.4f}')

    return best['macro'], best['threshold'], best['ml'], best['labels']

def grow_tree(data_q, target_data, thresholds, min_obs, max_regimes, verbose=True):
    """
    Grow the tree: at each step try splitting every leaf, apply the best (leaf, variable, threshold),
    stop when max_regimes is reached or no valid split exists.
    Returns (regime_labels, splits) where splits is a list of dicts describing each split.
    """
    n = len(data_q)
    regime_labels = np.zeros(n, dtype=int)
    splits = []
    next_label = 1

    while len(np.unique(regime_labels)) < max_regimes:
        if verbose:
            print(f'\n── Split {len(splits) + 1}: {len(np.unique(regime_labels))} leaf node(s) × '
                  f'{data_q.shape[1]} variables × {len(thresholds)} thresholds ──')

        best_ml, best_leaf, best_macro, best_threshold, best_labels = -np.inf, None, None, None, None
        for leaf in np.unique(regime_labels):
            if (regime_labels == leaf).sum() < 2 * min_obs:
                if verbose:
                    print(f'\nLeaf {leaf}: too small to split ({(regime_labels == leaf).sum()} observations), skipping.')
                continue
            macro, threshold, ml, labels = find_best_split(regime_labels, leaf, data_q, target_data, thresholds, min_obs, next_label, verbose)
            if macro is not None and ml > best_ml:
                best_ml, best_leaf, best_macro, best_threshold, best_labels = ml, leaf, macro, threshold, labels

        if best_macro is None:
            if verbose:
                print('\nNo valid split found anywhere. Stopping.')
            break

        regime_labels = best_labels
        splits.append({'split_number': len(splits) + 1, 'leaf_split': best_leaf, 'macro_variable': best_macro,
                       'threshold': best_threshold, 'log_ml': best_ml, 'new_label': next_label})
        if verbose:
            print(f'\nSplit {len(splits)} applied: Leaf {best_leaf} split on {best_macro} at {best_threshold:.1f}')
            for label in np.unique(regime_labels):
                print(f'    Regime {label}: {(regime_labels == label).sum()} months')
        next_label += 1

    return regime_labels, splits

def sort_by_spread(labels, index, yields_df):
    """
    Canonical relabelling by mean 10Y-2Y spread so labels are comparable across trees:
    0 = flattest curve, 1 = middle, 2 = steepest.
    """
    spread = (yields_df.loc[index, 'Y120M'] - yields_df.loc[index, 'Y24M']).values
    mean_spread = {label: spread[labels == label].mean() for label in np.unique(labels)}
    order = sorted(mean_spread, key=mean_spread.get)
    mapping = {old: new for new, old in enumerate(order)}
    return np.vectorize(mapping.get)(labels)

## 3. Full-Sample Tree

In [ ]:
print('Growing macro-instrumented regime tree...')
regime_labels, splits = grow_tree(data_q, target_data, thresholds, min_regime_months, max_regimes)
regime_labels_canonical = sort_by_spread(regime_labels, common_idx, yields_svensson)

print('\nTree Structure')
for s in splits:
    print(f"\nSplit {s['split_number']}: Regime {s['leaf_split']} split on {s['macro_variable']} at {s['threshold']}  (log ML = {s['log_ml']:.4f})")
    print(f"  Left child : Regime {s['leaf_split']}  ({s['macro_variable']} < {s['threshold']})")
    print(f"  Right child: Regime {s['new_label']}  ({s['macro_variable']} >= {s['threshold']})")

print('\nRegime sizes (raw label → canonical label):')
for label in sorted(np.unique(regime_labels)):
    mask = regime_labels == label
    print(f'  Regime {label} → canonical {regime_labels_canonical[mask][0]}: {mask.sum():4d} months ({mask.mean():.1%})')

## 4. Visualizations

In [ ]:
def shade_regimes(ax, dates, labels, colors, alpha=0.25):
    """Shade contiguous blocks of each regime label."""
    labels = np.asarray(labels)
    for label in np.unique(labels):
        in_block, block_start = False, None
        for i, date in enumerate(dates):
            if labels[i] == label and not in_block:
                in_block, block_start = True, date
            elif labels[i] != label and in_block:
                in_block = False
                ax.axvspan(block_start, date, alpha=alpha, color=colors[label], lw=0, zorder=0)
        if in_block:
            ax.axvspan(block_start, dates[-1], alpha=alpha, color=colors[label], lw=0, zorder=0)

def shade_recessions(ax, recession, alpha=0.35):
    """Shade NBER recessions (grey)."""
    in_rec, rec_start = False, None
    for date, val in recession['USREC'].items():
        if val == 1 and not in_rec:
            in_rec, rec_start = True, date
        elif val == 0 and in_rec:
            in_rec = False
            ax.axvspan(rec_start, date, alpha=alpha, color='grey', lw=0, zorder=1)
    if in_rec:
        ax.axvspan(rec_start, recession.index[-1], alpha=alpha, color='grey', lw=0, zorder=1)

def add_regime_legend(ax, labels, colors, loc='upper right'):
    for label in np.unique(labels):
        ax.scatter([], [], color=colors[label], alpha=0.5, label=f'Regime {label}', s=100, marker='s')
    ax.scatter([], [], color='grey', alpha=0.5, label='NBER Recession', s=100, marker='s')
    ax.legend(loc=loc, ncol=len(np.unique(labels)) + 1)

In [ ]:
# 4.1 Every variable (levels) shaded by raw regime
fig, axes = plt.subplots(len(all_cols), 1, figsize=(14, 3 * len(all_cols)))
for ax, col in zip(axes, all_cols):
    ax.plot(common_idx, data.loc[common_idx, col], color='black', lw=0.5, zorder=2)
    shade_regimes(ax, common_idx, regime_labels, regime_colors)
    shade_recessions(ax, recession)
    ax.set_xlim(common_idx.min() - pd.DateOffset(months=1), common_idx.max() + pd.DateOffset(months=1))
    ax.set_title(f'{col} with regime shading'); ax.set_ylabel(col)
add_regime_legend(axes[0], regime_labels, regime_colors)
plt.suptitle('Macro Variables by Detected Regime', y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
# 4.2 Yield curve shaded by canonical regime
plot_maturities = {'Y3M': '3M', 'Y6M': '6M', 'Y12M': '1Y', 'Y24M': '2Y', 'Y60M': '5Y', 'Y120M': '10Y'}
line_colors = ['#4575b4', '#74add1', '#abd9e9', '#fdae61', '#f46d43', '#d73027']

fig, ax = plt.subplots(figsize=(14, 5))
for (col, label), lc in zip(plot_maturities.items(), line_colors):
    ax.plot(common_idx, yields_svensson[col], color=lc, lw=1.2, label=label, zorder=3)
shade_regimes(ax, common_idx, regime_labels_canonical, regime_colors)
shade_recessions(ax, recession)
add_regime_legend(ax, regime_labels_canonical, regime_colors, loc='best')
ax.set_xlim(common_idx.min() - pd.DateOffset(months=1), common_idx.max() + pd.DateOffset(months=1))
ax.set_ylabel('Yield (%)'); ax.set_title('NSS Reconstructed Yields by Canonical Regime'); ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

## 5. Statistics

In [ ]:
# 5.1 Recession overlap
print(f'Total months: {len(common_idx)} | recession months: {rec_mask.sum()} ({rec_mask.mean():.1%})')
for label in sorted(np.unique(regime_labels)):
    mask = regime_labels == label
    overlap = (mask & rec_mask).sum()
    print(f'\nRegime {label} ({mask.sum()} months): {overlap} recession months '
          f'({overlap / mask.sum():.1%} of regime, {overlap / max(rec_mask.sum(), 1):.1%} of all recessions)')

In [ ]:
# 5.2 Mean and std of every variable (levels) within each regime
stat_rows = []
for label in sorted(np.unique(regime_labels)):
    mask = regime_labels == label
    row = {'Regime': label, 'N months': mask.sum()}
    for col in all_cols:
        vals = data.loc[common_idx[mask], col]
        row[f'{col} mean'] = round(vals.mean(), 3)
        row[f'{col} std'] = round(vals.std(), 3)
    stat_rows.append(row)
stats_df = pd.DataFrame(stat_rows).set_index('Regime')
stats_df

## 6. Save Full-Sample Results

In [ ]:
labels_df = pd.DataFrame({'regime_raw': regime_labels, 'regime_canonical': regime_labels_canonical}, index=common_idx)
labels_df.index.name = 'date'
labels_df.to_csv(f'{data_folder}/MacroRegimes3_Tree_Labels.csv')

results = pd.concat([data.loc[common_idx, all_cols], data_q.add_suffix('_q'), labels_df], axis=1)
results.to_csv(f'{data_folder}/MacroRegimes3_Tree_Results.csv')

pd.DataFrame(splits).to_csv(f'{data_folder}/MacroRegimes3_Tree_Splits.csv', index=False)
print('Saved MacroRegimes3_Tree_Labels.csv, MacroRegimes3_Tree_Results.csv, MacroRegimes3_Tree_Splits.csv')

## 7. Fold Trees for Out-of-Sample Evaluation

For every test year Y the PV-RF notebook needs two kinds of trees, both canonically ordered by mean spread:

- **evaluation tree** (`eval`): grown on all data up to the end of Y — the "ground truth" for Y's test months;
- **local training tree** (`local_<feature_set>`): grown on data up to `gap_months + 1` months before Y — what a
  real-time classifier could have known. Grown once per feature set: `all` (every variable) and `xbond` (no bond-market variables).

Folds where a tree finds fewer than `max_regimes` regimes are recorded with `n_regimes < 3` and skipped downstream.

In [ ]:
feature_sets = {'all': all_cols, 'xbond': [c for c in all_cols if c not in bond_cols]}

first_test_year = (common_idx.min() + pd.DateOffset(years=min_training_years)).year
last_test_year = common_idx.max().year

fold_labels, fold_splits = [], []

def record_fold(test_year, tree_type, idx, labels, splits):
    canonical = sort_by_spread(labels, idx, yields_svensson) if len(np.unique(labels)) > 1 else labels
    for date, raw, can in zip(idx, labels, canonical):
        fold_labels.append({'test_year': test_year, 'tree_type': tree_type, 'date': date, 'regime_raw': raw, 'regime_canonical': can})
    for s in splits:
        fold_splits.append({'test_year': test_year, 'tree_type': tree_type, 'n_regimes': len(np.unique(labels)), **s})
    if not splits:
        fold_splits.append({'test_year': test_year, 'tree_type': tree_type, 'n_regimes': 1})

for test_year in range(first_test_year, last_test_year + 1):
    test_start = pd.Timestamp(f'{test_year}-01-01')
    test_end = pd.Timestamp(f'{test_year}-12-01')
    train_end = test_start - pd.DateOffset(months=gap_months + 1)

    eval_idx = common_idx[common_idx <= test_end]
    train_idx = common_idx[common_idx <= train_end]
    if (common_idx >= test_start).sum() == 0:
        continue

    # Evaluation tree (always the full feature set)
    labels, sp = grow_tree(data_q.loc[eval_idx], yields_svensson.loc[eval_idx].values, thresholds, min_regime_months, max_regimes, verbose=False)
    record_fold(test_year, 'eval', eval_idx, labels, sp)
    msg = f'[{test_year}] eval tree ({len(eval_idx)} months): {len(np.unique(labels))} regimes'

    # Local training trees
    if len(train_idx) >= min_training_years * 12:
        for name, cols in feature_sets.items():
            labels, sp = grow_tree(data_q.loc[train_idx, cols], yields_svensson.loc[train_idx].values, thresholds, min_regime_months, max_regimes, verbose=False)
            record_fold(test_year, f'local_{name}', train_idx, labels, sp)
            msg += f' | local_{name} ({len(train_idx)} months): {len(np.unique(labels))} regimes'
    else:
        msg += f' | training data too short ({len(train_idx)} months), no local trees'
    print(msg)

fold_labels = pd.DataFrame(fold_labels)
fold_splits = pd.DataFrame(fold_splits)
fold_labels.to_csv(f'{data_folder}/MacroRegimes3_Tree_FoldLabels.csv', index=False)
fold_splits.to_csv(f'{data_folder}/MacroRegimes3_Tree_FoldSplits.csv', index=False)
print('\nSaved MacroRegimes3_Tree_FoldLabels.csv and MacroRegimes3_Tree_FoldSplits.csv')

In [ ]:
# Split-variable stability across folds: which variables did each tree split on?
def describe_splits(g):
    g = g.dropna(subset=['macro_variable']).sort_values('split_number')
    return ' → '.join(f"{r.macro_variable}@{r.threshold}" for r in g.itertuples()) or '(no split)'

rows = {}
for (year, tree_type), g in fold_splits.groupby(['test_year', 'tree_type']):
    rows.setdefault(year, {})[tree_type] = describe_splits(g)
split_table = pd.DataFrame(rows).T
print('Splits chosen by each fold tree:')
print(split_table.to_string())